# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Below, we list all record sets and their child fields and columns. Record sets are referenced by their `@id`.

In [ ]:
# Discover record sets and fields using mlcroissant API
record_sets = dataset.record_sets

print("Available record sets:\n")
for rs in record_sets:
    print(f"- Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    if fields:
        print(f"  Fields:")
        for fld in fields:
            if isinstance(fld, dict):
                print(f"    - {fld['@id']} ({fld.get('name', '')})")
            else:
                print(f"    - {fld}")
    if columns:
        print(f"  Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"    - {col['@id']} ({col.get('name', '')})")
            else:
                print(f"    - {col}")
    print()

# Print records from each record set using @id
for rs in record_sets:
    id_rs = rs['@id']
    print(f"Sample records in RecordSet {id_rs}:\n")
    for idx, rec in enumerate(dataset.records(record_set=id_rs)):
        print(rec)
        if idx == 2:
            break  # Only show a few records
    print()

## 3. Data Extraction
Load data from specified record sets into DataFrames for analysis.

Record sets, fields, and columns are referenced by their `@id`s.

In [ ]:
# Extract data from each record set
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_sets_ids:
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df

if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in RecordSet {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalizing numeric fields, and grouping by categorical fields.

Below, we select a numeric field (by its `@id`) and filter/group records.

In [ ]:
# Using the first record set and a numeric field
import numpy as np

if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    print(f"Working with RecordSet: {record_set_id}")
    # Find numeric fields (e.g., GAD-7, PHQ-9, PSQ scores)
    numeric_candidate_ids = [col for col in df.columns if any(x in col.lower() for x in ['score', 'gad', 'phq', 'psq'])]
    print(f"Numeric fields candidates: {numeric_candidate_ids}")

    if numeric_candidate_ids:
        numeric_field = numeric_candidate_ids[0]
        # Filter for values above threshold (example: 10)
        if df[numeric_field].dtype in [np.int64, np.float64] or df[numeric_field].apply(lambda x: str(x).isdigit()).all():
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = 10
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())
            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by categorical field (e.g., gender, education)
            group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'education', 'village', 'age', 'marital'])]
            if group_candidates:
                group_field = group_candidates[0]
                if group_field in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
                    print(grouped_df.head())
        else:
            print("Numeric field does not contain numeric data.")
    else:
        print("No candidate numeric fields found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions and relationships using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidate_ids:
    df = dataframes[record_set_id]
    numeric_field = numeric_candidate_ids[0]
    if df[numeric_field].dtype in [np.int64, np.float64]:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field], kde=True, bins=15)
        plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

    # Scatter plot against group field if exists
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['age'])]
    if group_candidates:
        group_field = group_candidates[0]
        plt.figure(figsize=(8,4))
        sns.scatterplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} vs {group_field} in RecordSet {record_set_id}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This exploratory notebook loaded and processed the Kilifi County mental health survey dataset using `mlcroissant`, referencing entities by their `@id`. We reviewed record sets, fields, extracted data, filtered and normalized scores, and grouped by demographics. The dataset supports diverse research and public health analyses leveraging clear semantic structure and open data standards.